In [ ]:
# ============ IMPORTS ============
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import quantstats as qs
from src.calculate_ml_metrics import calculate_ml_metrics
from src.calculate_trade_metrics import calculate_trade_metrics#
from xgboost import XGBClassifier

# ============ CONFIGURATION ============
feature_set = "all" # "all" or "confirmed" -  features confirmed by boruta algorithm
load_hyperparameters = True

# Fallback parameters — used only if load_hyperparameters = False
fallback_params = {
    "log_reg":        {"max_iter": 1000},
    "random_forest":  {"n_estimators": 200, "max_depth": 8},
    "svc":            {"kernel": "rbf"},
    "xgboost":        {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.05},
}
# =======================================

# ============ LOAD LABELLED DATA ============
labelled_data_cache = pd.read_pickle('data_cache/labelled_data.pkl')
labelled_df = labelled_data_cache['labelled_df']
X_train = labelled_data_cache['X_train']
X_test = labelled_data_cache['X_test']
y_train = labelled_data_cache['y_train']
y_test = labelled_data_cache['y_test']

# ============ LABEL REMAPPING ============
label_map   = {-1: 2, 0: 0, 1: 1}
label_unmap = { 2: -1, 0: 0, 1: 1}


# =========== LOAD LATEST FEATURE SELECTION FROM MLFLOW ============
fs_experiment  = mlflow.get_experiment_by_name("feature_selection")
# List Previous Runs
fs_runs = mlflow.search_runs(
            experiment_ids=[fs_experiment.experiment_id],
            filter_string = "status = 'FINISHED'",
            order_by = ['start_time DESC'])
# Load latest run ID
fs_run_id = fs_runs.iloc[0]["run_id"]
# Load artifact
fs_artifact_path = mlflow.artifacts.download_artifacts(
    run_id = fs_run_id, 
    artifact_path="feature_selection_status.pkl"
)
# Load dataframe
feature_selection_status = pd.read_pickle(fs_artifact_path)
# Load features
confirmed_features = feature_selection_status.loc[feature_selection_status['status'] == 'confirmed', 'feature'].tolist()
if feature_set == "confirmed":
    X_train = X_train[confirmed_features]
    X_test  = X_test[confirmed_features]

# ============ LOAD BEST HYPERPARAMETERS FROM MLFLOW ============
model_params = {name: dict(p) for name, p in fallback_params.items()}

if load_hyperparameters:
    hp_experiment = mlflow.get_experiment_by_name("hyperparameter_tuning")
    # List Previous Runs
    hp_runs = mlflow.search_runs(
        experiment_ids=[hp_experiment.experiment_id],
        filter_string="status = 'FINISHED'",
        order_by=["start_time DESC"]
    )

    for model_name in model_params:
        run_name_filter = f"{model_name}_{feature_set}_features"
        model_runs = hp_runs[hp_runs["tags.mlflow.runName"] == run_name_filter]

        if model_runs.empty:
            print(f"No tuning run found for {model_name}_{feature_set} — using fallback params.")
            continue

        best_run_id = model_runs.sort_values("metrics.best_cv_f1", ascending=False).iloc[0]["run_id"]
        artifact_path = mlflow.artifacts.download_artifacts(
            run_id=best_run_id,
            artifact_path=f"best_params_{model_name}_{feature_set}.pkl"
        )
        model_params[model_name] = pd.read_pickle(artifact_path).to_dict()

# ============ BUILD MODELS ============
models = {
    "log_reg": LogisticRegression(
        **model_params["log_reg"],
        random_state=42, class_weight="balanced", n_jobs=-1),
    "random_forest": RandomForestClassifier(
        **model_params["random_forest"],
        random_state=42, class_weight="balanced", n_jobs=-1),
    "svc": SVC(
        **model_params["svc"],
        probability=True, random_state=42, class_weight="balanced"),
    "xgboost": XGBClassifier(
        **model_params["xgboost"],
        random_state=42, eval_metric="mlogloss",
        tree_method="hist", device="cuda", n_jobs=-1),
}

In [ ]:
# =========== HELPER FUNCTION ===========
def get_evaluator_config(model_name):
    base = {"log_explainer": True,                      # Save explainer as MLflow artifact.
            "explainer_type": "auto",           
            # --- SHAP computation controls ---
            "explainability_nsamples": 200,             # Rows used for SHAP value estimation
            "explainability_kernel_link": "identity",   # identity, logit (log-odds)
            # --- Output controls ---
            "log_model_explanations": False,            # Log per-row SHAP values, computationally expensive
            "max_error_examples": 50,                   # How many misclassified examples to explain
    }
    if model_name == "log_reg":                         # Configure shap per model if needed.
        base["explainer_type"] = "linear"
    elif model_name == "svc":
        base["explainability_nsamples"] = 15
    return base

# =========== MLFLOW SETUP ============
mlflow.set_experiment("triple_barrier_classification") # or "bitcoin_near_peak_classification"
mlflow.sklearn.autolog()  # auto-log params, metrics, model, etc.

# =========== INITIALISE EVALUATION LIST
model_evaluation = []

# Get actual returns for BOTH train and test sets
actual_returns_train = labelled_df.loc[X_train.index, "Return_7day"]
actual_returns_test = labelled_df.loc[X_test.index, "Return_7day"]

# Initialise prediction dataframes
predictions_df_train = pd.DataFrame(index=X_train.index)
predictions_df_test = pd.DataFrame(index=X_test.index)

# ============ TRAIN MODELS ============
for model_name, model in models.items():
    with mlflow.start_run(run_name=f"{model_name}_{feature_set}_features"):
        # Train model
        if model_name == "xgboost": # XGBoost needs integer classes
            model.fit(X_train, y_train.map(label_map))
            train_pred = pd.Series(model.predict(X_train),index=X_train.index).map(label_unmap)
            test_pred = pd.Series(model.predict(X_test),index=X_test.index).map(label_unmap)
        else:
            model.fit(X_train, y_train)
            train_pred = model.predict(X_train)
            test_pred = model.predict(X_test)
        
# ============ GENERATE & STORE PREDICTIONS ============
        # Train predictions
        train_proba = model.predict_proba(X_train)
        predictions_df_train[f'pred_{model_name}'] = train_pred
        predictions_df_train[f'proba_{model_name}'] = list(train_proba)  # Store as list of arrays
        # Test predictions
        test_proba = model.predict_proba(X_test)
        predictions_df_test[f'pred_{model_name}'] = test_pred
        predictions_df_test[f'proba_{model_name}'] = list(test_proba)
        
        # Log the trained model with signature
        signature = infer_signature(X_train, train_pred)
        # defines the expected input and output formats for ML model, for reliability
        
        model_info = mlflow.sklearn.log_model(
            model, 
            artifact_path="model",
            signature=signature
        )
        
# ============ EVALUATE MODELS USING STORED PREDICTIONS ============
        # ML metrics (sklearn - consistent naming)
        train_ml_metrics = calculate_ml_metrics(
            y_train, 
            predictions_df_train[f'pred_{model_name}'], 
            predictions_df_train[f'proba_{model_name}'], 
            "train"
        )
        test_ml_metrics = calculate_ml_metrics(
            y_test, 
            predictions_df_test[f'pred_{model_name}'], 
            predictions_df_test[f'proba_{model_name}'], 
            "test"
        )
        
        # Trading metrics (QuantStats)
        train_trade_metrics = calculate_trade_metrics(
            y_train, 
            predictions_df_train[f'pred_{model_name}'], 
            actual_returns_train, 
            "train"
        )
        test_trade_metrics = calculate_trade_metrics(
            y_test, 
            predictions_df_test[f'pred_{model_name}'], 
            actual_returns_test, 
            "test"
        )

# ============ LOG ALL METRICS TO MLFLOW ============
        all_metrics = {
            **train_ml_metrics, 
            **test_ml_metrics,
            **train_trade_metrics, 
            **test_trade_metrics
        }
        
        for metric_name, metric_value in all_metrics.items():
            if metric_value is not None:
                mlflow.log_metric(metric_name, metric_value)
        
# ============ MLFLOW EVALUATE (for plots only) ============    
        # Create evaluation dataset (X_test + targets)
        eval_data = X_test.copy()
        if model_name == "xgboost":
            eval_data["Label_7day"] = y_test.map(label_map)
        else:
            eval_data["Label_7day"] = y_test
        
        # Evaluate with MLflow - generates all metrics & plots automatically
        eval_result = mlflow.models.evaluate(
            model=model_info.model_uri,
            data=eval_data,
            targets="Label_7day",
            model_type="classifier",
            evaluator_config=get_evaluator_config(model_name)
        )
        
# ============ FULL COMPARISON TABLE ============
        model_evaluation.append({
            "Model": model_name,
            # ML Metrics - Train
            "Train Acc": f"{train_ml_metrics['train_accuracy']:.3f}",
            "Train Prec": f"{train_ml_metrics['train_precision']:.3f}",
            "Train Rec": f"{train_ml_metrics['train_recall']:.3f}",
            "Train F1": f"{train_ml_metrics['train_f1_score']:.3f}",
            "Train ROC-AUC": f"{train_ml_metrics.get('train_roc_auc', 0):.3f}",
            # ML Metrics - Test
            "Test Acc": f"{test_ml_metrics['test_accuracy']:.3f}",
            "Test Prec": f"{test_ml_metrics['test_precision']:.3f}",
            "Test Rec": f"{test_ml_metrics['test_recall']:.3f}",
            "Test F1": f"{test_ml_metrics['test_f1_score']:.3f}",
            "Test ROC-AUC": f"{test_ml_metrics.get('test_roc_auc', 0):.3f}",
            # Trading Metrics - Train
            "Train WR": f"{train_trade_metrics['train_win_rate']:.3%}",
            "Train PF": f"{train_trade_metrics['train_profit_factor']:.3f}",
            "Train Avg Win%": f"{train_trade_metrics['train_avg_win_pct']:.3f}",
            "Train Avg Loss%": f"{train_trade_metrics['train_avg_loss_pct']:.3f}",
            "Train Max DD%": f"{train_trade_metrics['train_max_drawdown']:.3%}",
            "Train Sharpe": f"{train_trade_metrics['train_sharpe_ratio']:.3f}",
            "Train EV": f"{train_trade_metrics['train_expected_value']:.3f}",
            "Train Avg Loss Wrong%": f"{train_trade_metrics['train_avg_loss_when_wrong_pct']:.3f}",
            # Trading Metrics - Test
            "Test WR": f"{test_trade_metrics['test_win_rate']:.3%}",
            "Test PF": f"{test_trade_metrics['test_profit_factor']:.3f}",
            "Test Avg Win%": f"{test_trade_metrics['test_avg_win_pct']:.3f}",
            "Test Avg Loss%": f"{test_trade_metrics['test_avg_loss_pct']:.3f}",
            "Test Max DD%": f"{test_trade_metrics['test_max_drawdown']:.3%}",
            "Test Sharpe": f"{test_trade_metrics['test_sharpe_ratio']:.3f}",
            "Test EV": f"{test_trade_metrics['test_expected_value']:.3f}",
            "Test Avg Loss Wrong%": f"{test_trade_metrics['test_avg_loss_when_wrong_pct']:.3f}",
        })
        
# ============ LOG FEATURES  ============
        for feature in confirmed_features:
            mlflow.log_param(f"feature_{feature}", "Confirmed")
        
        if feature_set == "confirmed":
            final_ml_features = pd.Series(confirmed_features, name='feature')
        elif feature_set == "all":
            final_ml_features = feature_selection_status['feature']
        
        final_ml_features.to_pickle('data_cache/final_ml_features.pkl')
        mlflow.log_artifact('data_cache/final_ml_features.pkl')

# ============ STORE EVALUATIONS ============
model_evaluation = pd.DataFrame(model_evaluation)
model_evaluation.to_pickle('data_cache/model_evaluation.pkl')
mlflow.log_artifact('data_cache/model_evaluation.pkl')

# ============ DISPLAY RESULTS ============
print(model_evaluation.to_markdown(index=False))